In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 1. SEED SETTING
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# ==========================================
# 2. CUSTOM DATASET
# ==========================================
class DentalCariesDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'File_Path']
        label = int(self.df.loc[idx, 'Caries_Status'])
        
        image_cv = None
        if isinstance(img_path, str) and os.path.exists(img_path):
            image_cv = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
        if image_cv is None:
            image_cv = np.zeros((224, 224), dtype=np.uint8)
            
        image_equalized = self.clahe.apply(image_cv)
        image_rgb = cv2.cvtColor(image_equalized, cv2.COLOR_GRAY2RGB)
        image_pil = Image.fromarray(image_rgb)

        if self.transform:
            image_tensor = self.transform(image_pil)
        else:
            image_tensor = transforms.ToTensor()(image_pil)

        return image_tensor, torch.tensor(label, dtype=torch.float32).unsqueeze(0)

# ==========================================
# 3. CNN ARCHITECTURE
# ==========================================
class DentalCariesCNN(nn.Module):
    def __init__(self):
        super(DentalCariesCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(64, 128)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(self.relu3(self.fc1(x)))
        return self.sigmoid(self.fc2(x))

# ==========================================
# 4. MAIN PIPELINE
# ==========================================
def run_pipeline(csv_path, epochs=10):
    df = pd.read_csv(csv_path)
    df['Caries_Status'] = df['Clinical_Diagnosis'].apply(lambda x: 1 if x == 'Dental Caries' else 0)    
    # Calculate Weights for Class Imbalance
    neg = (df['Caries_Status'] == 0).sum()
    pos = (df['Caries_Status'] == 1).sum()
    weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32)

    train_df, test_df = train_test_split(df, test_size=0.3, stratify=df['Caries_Status'])
    
    t_transforms = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    v_transforms = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    train_loader = DataLoader(DentalCariesDataset(train_df, t_transforms), batch_size=32, shuffle=True)
    test_loader = DataLoader(DentalCariesDataset(test_df, v_transforms), batch_size=32)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DentalCariesCNN().to(device)
    
    # Corrected Weighted Loss Implementation
    criterion = nn.BCELoss(weight=weight.to(device).float())
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} Complete.")

    # Evaluation
    model.eval()
    all_preds, all_targs = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            preds = (model(images.to(device)) >= 0.5).cpu().numpy().astype(int)
            all_preds.extend(preds)
            all_targs.extend(labels.numpy().astype(int))
            
    print(classification_report(all_targs, all_preds, target_names=['Non-Caries', 'Caries']))

if __name__ == '__main__':
    run_pipeline(r"C:/Users/hp/Desktop/Final_Dataset_With_Paths.csv")

Epoch 1 Complete.
Epoch 2 Complete.
Epoch 3 Complete.
Epoch 4 Complete.
Epoch 5 Complete.
Epoch 6 Complete.
Epoch 7 Complete.
Epoch 8 Complete.
Epoch 9 Complete.
Epoch 10 Complete.
              precision    recall  f1-score   support

  Non-Caries       0.79      1.00      0.88       592
      Caries       0.00      0.00      0.00       158

    accuracy                           0.79       750
   macro avg       0.39      0.50      0.44       750
weighted avg       0.62      0.79      0.70       750



c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

: 